In [1]:
import pandas as pd
from datetime import datetime

In [3]:
path = "../MySQL_DataExtraction/medical_inventory.csv"
df = pd.read_csv(path, parse_dates = ["expiry_date", "order_date", "delivery_date", "movement_date"])
df

,medicine_name,category,Inventory Quatity,reorder_level,expiry_date,order_date,quantity_ordered,order_status,delivery_date,movement_date,movement_type,Stock Movement Quantity,supplier_name
0,Amoxicillin 500mg,Antibiotic,40,100,2026-12-15,2026-08-07,120,Delivered,2026-08-14,2026-08-07,ISSUE,80,HealthFirst Wholesale
1,Amoxicillin 500mg,Antibiotic,40,100,2026-12-15,2026-08-07,120,Delivered,2026-08-14,2026-09-15,ISSUE,60,HealthFirst Wholesale
2,Amoxicillin 500mg,Antibiotic,40,100,2026-12-15,2026-08-07,120,Delivered,2026-08-14,2026-09-07,ISSUE,65,HealthFirst Wholesale
3,Amoxicillin 500mg,Antibiotic,40,100,2026-12-15,2026-08-07,120,Delivered,2026-08-14,2026-08-15,ISSUE,55,HealthFirst Wholesale
4,Amoxicillin 500mg,Antibiotic,40,100,2026-12-15,2026-08-07,120,Delivered,2026-08-14,2026-10-15,ISSUE,70,HealthFirst Wholesale
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,Salbutamol 100mcg,Respiratory,75,50,2027-03-18,2026-09-02,60,Delivered,2026-09-09,2026-08-23,ISSUE,50,Ubuntu Medical Suppliers
68,Salbutamol 100mcg,Respiratory,75,50,2027-03-18,2026-09-02,60,Delivered,2026-09-09,2026-09-13,RECEIPT,35,Ubuntu Medical Suppliers
69,Salbutamol 100mcg,Respiratory,75,50,2027-03-18,2026-09-02,60,Delivered,2026-09-09,2026-09-23,ISSUE,45,Ubuntu Medical Suppliers
70,Salbutamol 100mcg,Respiratory,75,50,2027-03-18,2026-09-02,60,Delivered,2026-09-09,2026-10-13,RECEIPT,50,Ubuntu Medical Suppliers


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   medicine_name            72 non-null     str           
 1   category                 72 non-null     str           
 2   Inventory Quatity        72 non-null     int64         
 3   reorder_level            72 non-null     int64         
 4   expiry_date              72 non-null     datetime64[us]
 5   order_date               72 non-null     datetime64[us]
 6   quantity_ordered         72 non-null     int64         
 7   order_status             72 non-null     str           
 8   delivery_date            54 non-null     datetime64[us]
 9   movement_date            72 non-null     datetime64[us]
 10  movement_type            72 non-null     str           
 11  Stock Movement Quantity  72 non-null     int64         
 12  supplier_name            72 non-null     str     

# MEDICINES THAT NEED TO BE REORDERED

In [82]:
df["stock_status"] = df.apply(
    lambda row: "REORDER"
    if row["Inventory Quatity"] <= row["reorder_level"]
    else "SUFFICIENT",
    axis = 1
)

In [74]:
reorder_df = df[["medicine_name","Inventory Quatity", "reorder_level", "stock_status"]]
reorder_df

,medicine_name,Inventory Quatity,reorder_level,stock_status
0,Amoxicillin 500mg,40,100,REORDER
1,Amoxicillin 500mg,40,100,REORDER
2,Amoxicillin 500mg,40,100,REORDER
3,Amoxicillin 500mg,40,100,REORDER
4,Amoxicillin 500mg,40,100,REORDER
...,...,...,...,...
67,Salbutamol 100mcg,75,50,SUFFICIENT
68,Salbutamol 100mcg,75,50,SUFFICIENT
69,Salbutamol 100mcg,75,50,SUFFICIENT
70,Salbutamol 100mcg,75,50,SUFFICIENT


In [75]:
reorder_df.duplicated().sum()

np.int64(58)

In [91]:
reorder_df.drop_duplicates(inplace = True)

In [94]:
reorder_df.sort_values("stock_status")

,medicine_name,Inventory Quatity,reorder_level,stock_status
0,Amoxicillin 500mg,40,100,REORDER
12,Fluconazole 150mg,12,30,REORDER
15,Furosemide 40mg,65,70,REORDER
33,Insulin Regular,18,40,REORDER
39,Lisinopril 10mg,88,90,REORDER
42,Loratadine 10mg,22,60,REORDER
48,Metformin 500mg,25,100,REORDER
54,Omeprazole 20mg,55,70,REORDER
57,ORS Sachets,35,80,REORDER
6,Diclofenac 50mg,240,100,SUFFICIENT


In [92]:
reorder_df["stock_status"].value_counts()

stock_status
REORDER       9
SUFFICIENT    5
Name: count, dtype: int64

# MEDICINES EXPIRY DATE WARNING SIGN

In [104]:
today = datetime.today()

df["days_to_expiry"] = (df["expiry_date"] - today).dt.days

df["expiry_status"] = df["days_to_expiry"].apply(
    lambda days : "EXPIRED" 
    if days < 0
    else "URGENT" if days <= 30
    else "WARNING" if days <= 60
    else "SAFE"
)

In [107]:
expiry_df = df[["medicine_name", "days_to_expiry", "expiry_status"]]
expiry_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   medicine_name   72 non-null     str  
 1   days_to_expiry  72 non-null     int64
 2   expiry_status   72 non-null     str  
dtypes: int64(1), str(2)
memory usage: 3.2 KB


In [112]:
expiry_df.duplicated().sum()

np.int64(58)

In [117]:
expiry_df.drop_duplicates(inplace = True)
expiry_df.sort_values("expiry_status", ascending = False)

,medicine_name,days_to_expiry,expiry_status
0,Amoxicillin 500mg,36,WARNING
12,Fluconazole 150mg,3,URGENT
33,Insulin Regular,23,URGENT
42,Loratadine 10mg,9,URGENT
48,Metformin 500mg,16,URGENT
57,ORS Sachets,11,URGENT
6,Diclofenac 50mg,330,SAFE
15,Furosemide 40mg,80,SAFE
21,Gliclazide 80mg,279,SAFE
27,Ibuprofen 200mg,284,SAFE


# TARGET STOCK TO REODER

In [125]:
target_stock_units = 200

df["reorder_quantity"] = (
    target_stock_units - df["Inventory Quatity"]
).clip(lower = 0)

In [128]:
target_df = df[["medicine_name", "reorder_quantity"]]
target_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   medicine_name     72 non-null     str  
 1   reorder_quantity  72 non-null     int64
dtypes: int64(1), str(1)
memory usage: 2.3 KB


In [130]:
target_df.duplicated()

0     False
1      True
2      True
3      True
4      True
      ...  
67     True
68     True
69     True
70     True
71     True
Length: 72, dtype: bool

In [133]:
target_df.drop_duplicates(inplace = True)
target_df.sort_values("reorder_quantity")

,medicine_name,reorder_quantity
6,Diclofenac 50mg,0
27,Ibuprofen 200mg,0
60,Paracetamol 500mg,0
21,Gliclazide 80mg,95
39,Lisinopril 10mg,112
66,Salbutamol 100mcg,125
15,Furosemide 40mg,135
54,Omeprazole 20mg,145
0,Amoxicillin 500mg,160
57,ORS Sachets,165


# EXPORTING THE DATA

In [134]:
reorder_df.to_excel(
    "reorder_analysis.xlsx",
    index = False
)

expiry_df.to_excel(
    "expiry_analysis.xlsx",
    index = False
)

target_df.to_excel(
    "target_analysis.xlsx",
    index = False
)